# 臺灣 ERA5 → Cheng NN：新版五係數驗證入口

固定 1976–2025、50 年，21D、小模型。這是**臺灣真實資料試跑**，不是已知真值模擬或重新訓練。預設原正式模型；BOUND 必須明確指定，仍屬實驗，不會替換正式權重。

可用：地理篩選、輸入轉換、NN、support 檢查、五係數圖表，以及需另提供完整 predictors/config 的五係數 nested spatial FFS。NN 直接接空間選模，暫不執行獨立時間選模或變點檢定。尚待完成：候選變數對齊、buffer/fold 校準、空間模擬與 bootstrap。

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image

CURRENT = Path.cwd().resolve()
ROOT = next(p for p in (CURRENT, *CURRENT.parents) if (p / 'src/research_workflows.py').is_file())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from prepare_taiwan_cheng import prepare, DEFAULT_OUTPUT, DEFAULT_BOUNDARY
from infer_global_cheng_nn import DEFAULT_INPUT

MODEL = 'original'  # 'bound' 僅為實驗，不改正式模型
RUN_DIR = DEFAULT_OUTPUT / f'pilot_20261013_{MODEL}'
RUN_PREPARATION = False  # 已完成時直接看下方報告；重跑請改用全新 RUN_DIR


## 1. 地理與資料契約

Natural Earth `ADM0_A3=TWN` 的最大多邊形（臺灣本島）；格網中心落於多邊形且 `LSM > 0.5` 才保留。不是只切 bounding box，也不是沿用 TCCIP 的 1,385 格／8.5 km 規則。保留全球 GRID ID；逐年完整小時數、攝氏、正 IQR 檢查後，才轉為 EPSG:3826 公里座標。


In [ ]:
if RUN_PREPARATION:
    report = prepare(DEFAULT_INPUT, RUN_DIR, boundary=DEFAULT_BOUNDARY,
                     scope='main-island', model_kind=MODEL, device='auto')
else:
    print('不自動重跑；讀取已完成的獨立結果：', RUN_DIR)


## 2. 五係數結果與時間

輸出順序：`mu0, beta_mu, eta0, beta_sigma, xi0`；第三個欄位定義為 $\eta_0=\log\sigma_0$。BOUND 版在序列標準化尺度上由 $\delta$ 經 decoder 算出 $\eta_0=\log(A+e^\delta)$，再還原資料尺度；$\delta$ 不是額外的輸出係數或訓練標籤。全段 median/IQR 標準化後產生 21D；每段不分別標準化。BOUND decoder 額外使用同條 50 年觀測。

以下直方圖是 GRID 間估計值的分布，不是抽樣誤差或 bootstrap 分布；非零斜率不等於趨勢顯著。


In [ ]:
if (RUN_DIR / 'run_report.json').exists():
    report = json.loads((RUN_DIR / 'run_report.json').read_text(encoding='utf-8'))
    display(pd.Series({'GRID': report['n_grids'], **report['summary']}))
    display(pd.Series(report['timings'], name='seconds'))
    display(pd.read_csv(RUN_DIR / 'coefficient_summary.csv'))
    for filename in ('coefficient_maps.png', 'coefficient_histograms.png'):
        if (RUN_DIR / filename).exists():
            display(Image(filename=str(RUN_DIR / filename)))
else:
    print('尚無此 run 的結果。')


## 3. 五係數 FFS／nested buffered spatial CV

此工具使用**內層 buffered CV RMSE**選 predictors × kernel，外層只做 OOF 評估；不使用 AIC。固定 `full_no_selection`：保留完整五係數，NN 後不另做時間結構選擇或變點檢定。

先建立同一 GRID 的完整 predictors，並按 ERA5 臺灣格點密度重新設定 buffer/folds。`spatial_cv_config.example.json` 只是模板：高程不是最終候選清單，null buffer 必須先校準，範例 fold／min_train 也不是已驗證設定。程式會檢查所有內外層的保留格點數，不足就停止，不會自動縮小 buffer。


In [ ]:
CONFIG = ROOT / 'simulation/taiwan_validation/spatial_cv_config.example.json'
PREDICTORS = RUN_DIR / 'predictors_aligned.csv'  # 尚未生成：不得拿舊 TCCIP 表按列數硬接
SPATIAL_OUTPUT = RUN_DIR.parent / f'spatial_{MODEL}_new_run'
command = (f'uv run python .\\src\\cheng_spatial_cv.py --prepared "{RUN_DIR}" '
           f'--predictors "{PREDICTORS}" --config "{CONFIG}" --output "{SPATIAL_OUTPUT}"')
print('完成 predictors 與 buffer 校準後才執行：\n' + command)


## 4. 後续臺灣已知真值模擬／bootstrap

先完成候選變數對齊與空間校準，再保存五係數生成真值、設定跨 GRID 事件相依與 replicate 次數。每次重新生成 50 年資料後，重新標準化／21D／BOUND（若使用），用固定 NN 直接重跑空間 predictors/kernel 選擇。四種生成情境保留用於分組誤差評估，不作時間分類檢定。

實際 climate 真值未知；真實資料 OOF 與 NN 的差異只能稱 `RMSE_vs_NN_reference`。已知真值模擬另算五係數偏差/RMSE、RL 誤差、support 超界率及 predictors/kernel 選取頻率。結果條件於固定的線性時間模型。此 notebook 尚未執行上述重複模擬，也不輸出虛假的零誤差。